In [3]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from transformers import CLIPModel, CLIPProcessor

import medmnist
from medmnist import DermaMNIST, INFO

warnings.filterwarnings("ignore")

# -----------------------------
# 1) Dynamic runtime configuration
# -----------------------------
IS_KAGGLE = os.path.exists("/kaggle/working")

if IS_KAGGLE:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    BATCH_SIZE = 32
    IMAGE_SIZE = 224
    EPOCHS = 5
    DEBUG_SUBSET = False
else:
    DEVICE = "cpu"
    BATCH_SIZE = 1
    IMAGE_SIZE = 224
    EPOCHS = 1
    DEBUG_SUBSET = True

DATA_ROOT = Path("./data")
ARTIFACT_DIR = Path("./artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_NAME = "openai/clip-vit-base-patch32"
NUM_WORKERS = 0
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"Device: {DEVICE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")
print(f"Debug subset: {DEBUG_SUBSET}")


# -----------------------------
# 2) Dataset ingestion and pipeline
# -----------------------------
class CLIPImageDataset(Dataset):
    def __init__(self, dataset, processor):
        self.dataset = dataset
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        image = Image.fromarray(np.asarray(image)).convert("RGB")
        processed = self.processor(images=image, return_tensors="pt")
        pixel_values = processed["pixel_values"].squeeze(0)
        return pixel_values, int(label)


def load_dermamnist_datasets():
    train_dataset = DermaMNIST(split="train", download=True, root=str(DATA_ROOT))
    test_dataset = DermaMNIST(split="test", download=True, root=str(DATA_ROOT))

    if DEBUG_SUBSET:
        train_dataset.imgs = train_dataset.imgs[:10]
        train_dataset.labels = train_dataset.labels[:10]
        test_dataset.imgs = test_dataset.imgs[:5]
        test_dataset.labels = test_dataset.labels[:5]
        train_dataset.info["n_samples"]["train"] = len(train_dataset.imgs)
        test_dataset.info["n_samples"]["test"] = len(test_dataset.imgs)

    return train_dataset, test_dataset


train_dataset, test_dataset = load_dermamnist_datasets()
info = INFO["dermamnist"]
class_names = [str(name) for name in info["label"]]
num_classes = len(class_names)
print(f"Dataset loaded: train={len(train_dataset)}, test={len(test_dataset)}")
print(f"Classes: {class_names}")


# -----------------------------
# 3) Zero-shot CLIP baseline
# -----------------------------
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model = CLIPModel.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()


def _extract_embedding(output):
    if torch.is_tensor(output):
        return output
    if hasattr(output, "text_embeds") and output.text_embeds is not None:
        return output.text_embeds
    if hasattr(output, "image_embeds") and output.image_embeds is not None:
        return output.image_embeds
    if hasattr(output, "pooler_output") and output.pooler_output is not None:
        return output.pooler_output
    if isinstance(output, (tuple, list)) and len(output) > 0 and torch.is_tensor(output[0]):
        return output[0]
    raise TypeError(f"Unsupported CLIP output type: {type(output)!r}")


def run_zero_shot_evaluation(dataset, class_names, device):
    prompts = [f"A clinical image of {name}" for name in class_names]
    text_inputs = processor(text=prompts, padding=True, return_tensors="pt")
    text_inputs = {k: v.to(device) for k, v in text_inputs.items()}

    with torch.no_grad():
        text_outputs = model.get_text_features(**text_inputs)
        text_features = _extract_embedding(text_outputs)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    labels = []
    features = []
    for image, label in dataset:
        image = Image.fromarray(np.asarray(image)).convert("RGB")
        image_inputs = processor(images=image, return_tensors="pt")
        image_inputs = {k: v.to(device) for k, v in image_inputs.items()}
        with torch.no_grad():
            image_outputs = model.get_image_features(**image_inputs)
            image_feature = _extract_embedding(image_outputs)
            image_feature = image_feature / image_feature.norm(dim=-1, keepdim=True)
        features.append(image_feature.cpu())
        labels.append(int(label))

    image_features = torch.cat(features, dim=0)
    logits = 100.0 * image_features @ text_features.cpu().T
    probs = torch.softmax(logits, dim=-1).numpy()
    preds = probs.argmax(axis=-1)
    labels = np.array(labels)

    acc = accuracy_score(labels, preds)
    try:
        auc = roc_auc_score(labels, probs, multi_class="ovr", average="macro")
    except ValueError:
        auc = float("nan")

    return preds, probs, acc, auc


zero_shot_preds, zero_shot_probs, zero_shot_acc, zero_shot_auc = run_zero_shot_evaluation(
    test_dataset, class_names, DEVICE
)
print(f"Zero-shot ACC: {zero_shot_acc:.4f}")
print(f"Zero-shot Macro AUC: {zero_shot_auc:.4f}")


# -----------------------------
# 4) Linear probing with frozen CLIP backbone
# -----------------------------
class CLIPLinearProber(nn.Module):
    def __init__(self, clip_model, num_classes):
        super().__init__()
        self.clip_model = clip_model
        for param in self.clip_model.parameters():
            param.requires_grad_(False)

        hidden_size = getattr(self.clip_model.config, "projection_dim", None)
        if hidden_size is None:
            hidden_size = self.clip_model.config.vision_config.hidden_size
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes),
        )

    def forward(self, pixel_values):
        image_features = self.clip_model.get_image_features(pixel_values=pixel_values)
        image_features = _extract_embedding(image_features)
        return self.head(image_features)


train_loader = DataLoader(
    CLIPImageDataset(train_dataset, processor),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
)

test_loader = DataLoader(
    CLIPImageDataset(test_dataset, processor),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)


def train_linear_prober(model, train_loader, test_loader, device, epochs):
    prober = CLIPLinearProber(model, num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(prober.head.parameters(), lr=1e-3)

    for epoch in range(epochs):
        prober.train()
        running_loss = 0.0
        for pixel_values, labels in train_loader:
            pixel_values = pixel_values.to(device)
            labels = labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = prober(pixel_values)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        avg_loss = running_loss / max(1, len(train_loader))
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {avg_loss:.4f}")

    prober.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for pixel_values, labels in test_loader:
            pixel_values = pixel_values.to(device)
            logits = prober(pixel_values)
            preds = logits.argmax(dim=-1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels.tolist())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    acc = accuracy_score(all_labels, all_preds)

    probs = np.zeros((len(all_labels), num_classes))
    if len(all_labels) > 0:
        logits_test = []
        with torch.no_grad():
            for pixel_values, _ in test_loader:
                pixel_values = pixel_values.to(device)
                logits_test.append(prober(pixel_values).cpu())
        probs = torch.softmax(torch.cat(logits_test, dim=0), dim=-1).numpy()

    if len(np.unique(all_labels)) >= 2 and len(np.unique(all_labels)) == num_classes:
        try:
            auc = roc_auc_score(all_labels, probs, multi_class="ovr", average="macro")
        except ValueError:
            auc = float("nan")
    else:
        auc = float("nan")

    checkpoint = {
        "model_state_dict": prober.state_dict(),
        "class_names": class_names,
    }
    torch.save(checkpoint, ARTIFACT_DIR / "clip_linear_prober.pt")
    return prober, acc, auc


prober, linear_acc, linear_auc = train_linear_prober(model, train_loader, test_loader, DEVICE, EPOCHS)
print(f"Linear probing ACC: {linear_acc:.4f}")
print(f"Linear probing Macro AUC: {linear_auc:.4f}")


# -----------------------------
# 5) Scholarly metrics and analysis compliance
# -----------------------------
print("Metrics completed for Zero-Shot and Linear Probing.")


# -----------------------------
# 6) Evaluation inference workflow
# -----------------------------
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


def predict_for_evaluation(image_folder_path, output_csv_path):
    image_folder = Path(image_folder_path)
    if not image_folder.exists() or not image_folder.is_dir():
        raise FileNotFoundError(f"Image folder does not exist: {image_folder}")

    image_paths = sorted(
        [p for p in image_folder.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS]
    )
    if not image_paths:
        raise ValueError(f"No supported image files found in {image_folder}")

    model_for_eval = CLIPModel.from_pretrained(MODEL_NAME)
    model_for_eval.to(DEVICE)
    model_for_eval.eval()

    prober_for_eval = CLIPLinearProber(model_for_eval, num_classes).to(DEVICE)
    checkpoint_path = ARTIFACT_DIR / "clip_linear_prober.pt"
    if checkpoint_path.exists():
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
        prober_for_eval.load_state_dict(checkpoint["model_state_dict"])
    prober_for_eval.eval()

    image_ids = []
    class_ids = []

    with torch.no_grad():
        for image_path in image_paths:
            image = Image.open(image_path).convert("RGB")
            inputs = processor(images=image, return_tensors="pt")
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            logits = prober_for_eval(inputs["pixel_values"])
            pred_class_id = int(logits.argmax(dim=-1).item())
            image_ids.append(image_path.stem)
            class_ids.append(pred_class_id)

    output_df = pd.DataFrame({"image_id": image_ids, "class_id": class_ids})
    output_df.to_csv(output_csv_path, index=False)
    print(f"Saved evaluation CSV to {output_csv_path}")
    return output_df


Environment: Local
Device: cpu
Batch size: 1
Epochs: 1
Debug subset: True
Dataset loaded: train=10, test=5
Classes: ['0', '1', '2', '3', '4', '5', '6']


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Zero-shot ACC: 0.2000
Zero-shot Macro AUC: nan
Epoch 1/1 - Loss: 1.3768
Linear probing ACC: 0.4000
Linear probing Macro AUC: nan
Metrics completed for Zero-Shot and Linear Probing.
